# User study: rule-based vs weight-based explanations across two datasets, simulated with CoXAM

Same four phases as the general tutorial, with two substitutions: the virtual
participant is **CoXAM**, a cognitive model fitted to real study participants
rather than a stand-in classifier, and the study runs across **two datasets**
(`wine_quality` and `mushrooms`) as a between-subjects factor.

The study asks whether people predict an AI's output more accurately from a
**rule-based** explanation (`decision_tree`) or a **weight-based** one
(`logistic_regression`), whether having any explanation helps at all
(`tested_w_xai`), and whether either effect replicates across two different
datasets (`dataset`).

Everything is written out step by step as plain API calls, so the notebook runs
top to bottom in Colab with nothing to upload and no design file to load.

### Three things this notebook is strict about

**1. The IV must be named `xai_type`.** CoXAM reads the explanation family from
`xai_type` (and the per-trial `shown_xai_type` derived from it). Name the same
IV `xai_method` and CoXAM sees no condition at all: every trial silently falls
back to a default, and your rules-vs-weights comparison becomes two samples from
one distribution. The analysis still runs and still prints a p-value -- it is
just measuring nothing.

**2. No AI model is trained.** Both datasets are covered by CoXAM's published
corpus, so `run_coxam_study(source="assets")` reads the corpus's own AI
predictions and DT/LR surrogates. Training an MLP here would be a long run
producing a model nothing downstream reads -- which is exactly why the server
skips it too.

**3. Instance ids are dataset-local.** Each dataset numbers its own instances
from zero, so trials must be restricted to each dataset's *own* corpus ids
separately. Pooling them into one list would let ids valid only for the larger
dataset leak into the smaller one's pool.

## Phase 0: Set up your environment

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Walk up until the repo root (the directory holding `src/`) is found, so the
# notebook runs from anywhere inside the checkout.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "src").is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import src as xk

OUTPUT_DIR = REPO_ROOT / "tutorials" / "coxam_workflow_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("repo root :", REPO_ROOT)
print("output dir:", OUTPUT_DIR)

## Phase 1: Configure the User Study

### Step 1.1 Initialize the study workflow

**Goal:**

Instantiate the study object that orchestrates the pipeline and stores its
artifacts.

**Instruction:**

Create a study workflow instance with a project name and `OUTPUT_DIR`.

<details>
<summary>Hint</summary>

Call

```python
xk.xaikitTest(project_name: str, output_dir: str | Path)
```

</details>

In [ ]:
# YOUR CODE HERE
# xaikitTest = ?

In [ ]:
# Delete this cell after passing all tests. This is NOT for interviewees

xaikitTest = xk.xaikitTest("coxam_rules_weights_study", output_dir=OUTPUT_DIR)
xaikitTest

<details>
<summary>Click here to see the correct code</summary>

```python
xaikitTest = xk.xaikitTest("coxam_rules_weights_study", output_dir=OUTPUT_DIR)
xaikitTest
```

</details>

### Step 1.2 Define the user study variables

**Goal:**

Register the study's variables and validate the design.

**Instruction:**

*   `dataset` -- which dataset a participant is assigned to. **Between**-subjects,
    levels `wine_quality` and `mushrooms`. A participant sees one dataset for
    the whole study; the name `dataset` is what the trial generator and the
    analysis both key off.
*   `xai_type` -- the explanation family. Within-subjects, block-randomized,
    levels `decision_tree` and `logistic_regression`. **The name matters**: see
    the note at the top of the notebook.
*   `tested_w_xai` -- whether the trial shows an explanation. Within-subjects,
    trial-randomized, levels `True` and `False`.
*   `age_group` -- a control variable with levels `young` and `adult`.
*   `forward_accuracy` -- the dependent variable, `continuous`.

<details>
<summary>Hint</summary>

*   Same shape as the general tutorial: build `iv_config`, `cvs` and `dvs`, then
    `xaikitTest.set_design(...)` and `xaikitTest.validate(stage="design", ...)`.
*   Use the key `"xai_type"`, not `"xai_method"`.
*   A between-subjects entry needs no `randomization` -- only within-subjects
    IVs are counterbalanced within a participant.

</details>

In [ ]:
# YOUR CODE HERE

## Construct IVs, DVs, CVs
iv_config = {}

cvs = {}

dvs = {}

## Register

## Validate

In [ ]:
# Delete this cell after passing all tests. This is NOT for interviewees

iv_config = {
    "dataset": {
        "type": "between",
        "levels": ["wine_quality", "mushrooms"],
    },
    "xai_type": {
        "type": "within",
        "randomization": "block",
        "levels": ["decision_tree", "logistic_regression"],
    },
    "tested_w_xai": {
        "type": "within",
        "randomization": "trial",
        "levels": [True, False],
    },
}

cvs = {"age_group": ["young", "adult"]}

dvs = {"forward_accuracy": ["continuous"]}

xaikitTest.set_design(iv_config=iv_config, cvs=cvs, dvs=dvs, show=True)
xaikitTest.validate(stage="design", strict=False, show=True)

<details>
<summary>Click here to see the correct code</summary>

```python
iv_config = {
    "dataset": {
        "type": "between",
        "levels": ["wine_quality", "mushrooms"],
    },
    "xai_type": {
        "type": "within",
        "randomization": "block",
        "levels": ["decision_tree", "logistic_regression"],
    },
    "tested_w_xai": {
        "type": "within",
        "randomization": "trial",
        "levels": [True, False],
    },
}

cvs = {"age_group": ["young", "adult"]}

dvs = {"forward_accuracy": ["continuous"]}

xaikitTest.set_design(iv_config=iv_config, cvs=cvs, dvs=dvs, show=True)
xaikitTest.validate(stage="design", strict=False, show=True)
```

</details>

### Step 1.3 Configure the datasets

**Goal:**

Prepare **both** datasets the study runs on.

Passing a list rather than a single id is the whole multi-dataset switch: it
returns a `{dataset_id: prepared}` registry, stores it on the study, and keeps
each dataset's features and instance-id space separate. Nothing is merged --
one dataset's instance ids mean nothing in another's feature space.

`cognitive_model_id="coxam"` matters too: it tells `prepare_dataset` to select
and order features the way CoXAM's corpus expects, rather than leaving the
choice to the generic default.

Run the code below.

In [ ]:
data_by_dataset = xaikitTest.prepare_dataset(
    ["wine_quality", "mushrooms"],
    use_default_features=True,
    show_available=False,
    show_summary=True,
    cognitive_model_id="coxam",
)

for dataset_id, prepared in data_by_dataset.items():
    print(f"{dataset_id:<14} features={prepared.raw_feature_names}")
    print(f"{'':<14} train={prepared.X_train.shape} test={prepared.X_test.shape}")

### Step 1.4 Restrict each dataset to what CoXAM's corpus can serve

**Goal:**

Narrow each dataset's train/test split down to the instances CoXAM's published
corpus actually shipped.

**Instruction:**

`prepare_dataset` splits the *raw* dataset, which knows nothing about the
corpus. CoXAM's corpus carries only a subset of instances -- so a trial
referencing an id outside it fails at simulation time, long after the trial
table looked fine.

Filter both splits, **per dataset, against that dataset's own corpus ids**.
Instance ids are dataset-local, so never pool the two id sets into one list.

Print how many ids survive in each split so you can see the effect.

<details>
<summary>Hint</summary>

*   `from src.virtual_experiment_executor.experiment_simualtion.CoXAM.coxam_trial_executor import coxam_available_instance_ids`
*   `coxam_available_instance_ids(dataset_id)` gives the ids that dataset's
    corpus can serve.
*   Loop over `data_by_dataset.items()` and rewrite
    `prepared.split.train_instance_ids` / `prepared.split.test_instance_ids`,
    keeping only ids in that dataset's corpus set. `np.asarray(sorted(...))` is
    the shape they need to stay in.

</details>

In [ ]:
# YOUR CODE HERE

In [ ]:
# Delete this cell after passing all tests. This is NOT for interviewees

from src.virtual_experiment_executor.experiment_simualtion.CoXAM.coxam_trial_executor import (
    coxam_available_instance_ids,
)

for dataset_id, prepared in data_by_dataset.items():
    corpus_ids = set(coxam_available_instance_ids(dataset_id))
    prepared.split.train_instance_ids = np.asarray(
        sorted(i for i in prepared.split.train_instance_ids.tolist() if i in corpus_ids)
    )
    prepared.split.test_instance_ids = np.asarray(
        sorted(i for i in prepared.split.test_instance_ids.tolist() if i in corpus_ids)
    )
    print(f"{dataset_id:<14} corpus={len(corpus_ids):<5} "
          f"train={len(prepared.split.train_instance_ids):<5} "
          f"test={len(prepared.split.test_instance_ids)}")

<details>
<summary>Click here to see the correct code</summary>

```python
from src.virtual_experiment_executor.experiment_simualtion.CoXAM.coxam_trial_executor import (
    coxam_available_instance_ids,
)

for dataset_id, prepared in data_by_dataset.items():
    corpus_ids = set(coxam_available_instance_ids(dataset_id))
    prepared.split.train_instance_ids = np.asarray(
        sorted(i for i in prepared.split.train_instance_ids.tolist() if i in corpus_ids)
    )
    prepared.split.test_instance_ids = np.asarray(
        sorted(i for i in prepared.split.test_instance_ids.tolist() if i in corpus_ids)
    )
    print(f"{dataset_id:<14} corpus={len(corpus_ids):<5} "
          f"train={len(prepared.split.train_instance_ids):<5} "
          f"test={len(prepared.split.test_instance_ids)}")
```

</details>

### Step 1.5 Generate counterbalanced trials

**Goal:**

Check the condition cells divide evenly, then generate the counterbalanced trial
table across both datasets.

**Instruction:**

*   Compute the number of balanced cells from the two **within**-subjects IVs
    and assert that `num_testing` divides evenly by it. The between-subjects
    `dataset` IV does not enter this count -- it splits participants, not a
    participant's own trials.
*   Generate trials for 4 participants per between-subjects condition, with 4
    training and 8 testing trials each.
*   Show the per-cell counts, and confirm each participant stays on one dataset.

<details>
<summary>Hint</summary>

*   `n_balanced_cells = len(iv_config["xai_type"]["levels"]) * len(iv_config["tested_w_xai"]["levels"])`
*   `xaikitTest.generate_trials(participants_per_between_condition=..., num_training=..., num_testing=..., output_dir="trials", show=True)`
*   The trial table gains a `dataset` column (the condition) and a
    `shown_xai_type` column (the family that trial actually displays -- what
    CoXAM reads).
*   Check assignment with `trials_df.groupby("participantId")["dataset"].nunique().max()`.

</details>

In [ ]:
# YOUR CODE HERE

In [ ]:
# Delete this cell after passing all tests. This is NOT for interviewees

n_block_conditions = len(iv_config["xai_type"]["levels"])
n_trial_conditions = len(iv_config["tested_w_xai"]["levels"])
n_balanced_cells = n_block_conditions * n_trial_conditions

participants_per_between_condition = 4
num_training = 4
num_testing = 8

assert num_testing % n_balanced_cells == 0, (
    "num_testing must divide evenly across block x trial-level cells: "
    f"{num_testing} trials for {n_balanced_cells} cells"
)

trial_result = xaikitTest.generate_trials(
    participants_per_between_condition=participants_per_between_condition,
    num_training=num_training,
    num_testing=num_testing,
    output_dir="trials",
    show=True,
)

trials_df = pd.DataFrame(trial_result.trials)
print("shown_xai_type present:", "shown_xai_type" in trials_df.columns)
print("datasets per participant (must be 1):",
      trials_df.groupby("participantId")["dataset"].nunique().max())

display(
    trials_df
    .groupby(["dataset", "xai_type", "tested_w_xai"])
    .size()
    .reset_index(name="trials")
)
trials_df.head(12)

<details>
<summary>Click here to see the correct code</summary>

```python
n_block_conditions = len(iv_config["xai_type"]["levels"])
n_trial_conditions = len(iv_config["tested_w_xai"]["levels"])
n_balanced_cells = n_block_conditions * n_trial_conditions

participants_per_between_condition = 4
num_training = 4
num_testing = 8

assert num_testing % n_balanced_cells == 0, (
    "num_testing must divide evenly across block x trial-level cells: "
    f"{num_testing} trials for {n_balanced_cells} cells"
)

trial_result = xaikitTest.generate_trials(
    participants_per_between_condition=participants_per_between_condition,
    num_training=num_training,
    num_testing=num_testing,
    output_dir="trials",
    show=True,
)

trials_df = pd.DataFrame(trial_result.trials)
print("shown_xai_type present:", "shown_xai_type" in trials_df.columns)
print("datasets per participant (must be 1):",
      trials_df.groupby("participantId")["dataset"].nunique().max())

display(
    trials_df
    .groupby(["dataset", "xai_type", "tested_w_xai"])
    .size()
    .reset_index(name="trials")
)
trials_df.head(12)
```

</details>

## Phase 2: Prepare the AI Model and its Explanations

**Skipped -- and that is the correct configuration here, not a shortcut.**

Both `wine_quality` and `mushrooms` are covered by CoXAM's published corpus.
`run_coxam_study(source="assets")` reads that corpus's own AI predictions and
its own decision-tree / logistic-regression surrogates. It never touches
`study.trained_ai_model`, and it never reads an explanation table you build
here. Training an MLP would be a long run producing a model nothing downstream
looks at.

There is also a structural reason a multi-dataset study cannot train one:
`train_AI_model()` trains against a single `study.data`, and this study has two
prepared datasets with different feature spaces. One model cannot serve both.

The general tutorial's Phase 2 shows the full train-and-explain path for the
case where you *do* need it.

### Step 2.1 Name the model the corpus labelled its predictions with

**Goal:**

Tell the study which model name to look predictions up under.

**Instruction:**

`train_AI_model()` would normally set `study.model_name` as a side effect. It
was skipped, so nothing set it -- and CoXAM looks the corpus's AI predictions up
*by* model name. Left unset it searches for a model that was never trained and
fails with "No instances with both features and a labelled AI prediction".

The corpus labelled its predictions `"mlp"`. Set that, then confirm it.

<details>
<summary>Hint</summary>

*   It is a plain attribute assignment: `xaikitTest.model_name = "mlp"`.
*   This is the one line of Phase 2 that still matters when training is skipped.

</details>

In [ ]:
# YOUR CODE HERE

In [ ]:
# Delete this cell after passing all tests. This is NOT for interviewees

xaikitTest.model_name = "mlp"
print("model name:", xaikitTest.model_name)

<details>
<summary>Click here to see the correct code</summary>

```python
xaikitTest.model_name = "mlp"
print("model name:", xaikitTest.model_name)
```

</details>

## Phase 3: Run the User Study Simulation

### Step 3.1 Register CoXAM as the cognitive model

**Goal:**

Select the real fitted cognitive model as the virtual participant.

**Instruction:**

Register CoXAM by **id**, then confirm it took.

Registering by id is what routes `run_experiment` into CoXAM's own runner. If
you instead hand `set_cognitive_model` a plain callable -- such as one of the
`study_simulators` helpers -- the study runs the generic executor and CoXAM is
never involved, however the notebook is titled.

<details>
<summary>Hint</summary>

*   `xaikitTest.set_cognitive_model(cognitive_model_id="coxam")`
*   Read back `xaikitTest.cognitive_model_id`.

</details>

In [ ]:
# YOUR CODE HERE

In [ ]:
# Delete this cell after passing all tests. This is NOT for interviewees

xaikitTest.set_cognitive_model(cognitive_model_id="coxam")
print(f"Cognitive model: {xaikitTest.cognitive_model_id}")

<details>
<summary>Click here to see the correct code</summary>

```python
xaikitTest.set_cognitive_model(cognitive_model_id="coxam")
print(f"Cognitive model: {xaikitTest.cognitive_model_id}")
```

</details>

### Step 3.2 Run the virtual study simulation

**Goal:**

Run every simulated participant through every trial with CoXAM.

**Instruction:**

*   Run the whole experiment.
*   Pass `source="assets"` so CoXAM reads the published corpus's own AI
    predictions and surrogates for each dataset.
*   Keep the testing rows separately, and show the per-condition counts.

Two things happen here that are worth knowing about. There is no
`explanation_pool` argument -- CoXAM builds its own surrogates internally, so
an explanation table is not what it reads. And because this study has two
prepared datasets, `run_experiment` loops the runner once per dataset behind the
scenes, pointing it at that dataset's own data and trial rows, then tags and
concatenates the results. The runner itself never sees two datasets at once.

<details>
<summary>Hint</summary>

*   `xaikitTest.run_experiment(mode="whole_experiment", participant_id=None, source="assets")`
*   Group by `["phase", "dataset", "xai_type", "tested_w_xai"]`.

</details>

In [ ]:
# YOUR CODE HERE

In [ ]:
# Delete this cell after passing all tests. This is NOT for interviewees

simulated_results = xaikitTest.run_experiment(
    mode="whole_experiment",
    participant_id=None,
    source="assets",
)
testing_results = simulated_results.query("phase == 'testing'").copy()

print(f"{len(xaikitTest.trials):,} trials -> {len(simulated_results):,} recorded rows")
print("datasets in results:", sorted(simulated_results["dataset"].unique()))

display(
    simulated_results
    .groupby(["phase", "dataset", "xai_type", "tested_w_xai"], dropna=False)
    .size()
    .reset_index(name="rows")
)
simulated_results.head()

<details>
<summary>Click here to see the correct code</summary>

```python
simulated_results = xaikitTest.run_experiment(
    mode="whole_experiment",
    participant_id=None,
    source="assets",
)
testing_results = simulated_results.query("phase == 'testing'").copy()

print(f"{len(xaikitTest.trials):,} trials -> {len(simulated_results):,} recorded rows")
print("datasets in results:", sorted(simulated_results["dataset"].unique()))

display(
    simulated_results
    .groupby(["phase", "dataset", "xai_type", "tested_w_xai"], dropna=False)
    .size()
    .reset_index(name="rows")
)
simulated_results.head()
```

</details>

### Step 3.3 Inspect CoXAM's own columns

A baseline classifier returns a prediction and nothing else. CoXAM also reports
*how* it got there, which is most of the point of using a cognitive model.

Run the cell below to see the strategy it selected per trial and how often its
answer matched the AI.

In [ ]:
coxam_columns = [
    "participantId", "dataset", "xai_type", "shown_xai_type", "tested_w_xai",
    "explanation_type", "selected_strategy", "ai_prediction",
    "agent_prediction", "cognitive_correct_vs_ai", "forward_accuracy",
]
available = [column for column in coxam_columns if column in testing_results.columns]
display(testing_results[available].head(10))

print("\nStrategy selected, by explanation family:")
display(
    testing_results
    .groupby(["xai_type", "selected_strategy"], dropna=False)
    .size()
    .reset_index(name="trials")
)

### Step 3.4 Save the simulation results

In [ ]:
csv_path, json_path = xaikitTest.save_results(out_dir="simulated_results")

print(csv_path)
print(json_path)

## Phase 4: Analysis of Simulation Results

### Step 4.1 Descriptive statistics and the results grid

**Goal:**

Summarise performance per condition and render the results grid.

**Instruction:**

*   Run the descriptive analysis of `xai_type` against `forward_accuracy`, and
    again for `dataset`, displaying each `descriptives` table.
*   Render the grid across all three IVs and the DV, testing phase only.

`dataset` is an ordinary between-subjects IV here, so it needs no special
handling -- it flows into the same analysis and the same grid as the other two.

<details>
<summary>Hint</summary>

*   `xaikitTest.analyze_iv_dv(iv="xai_type", dv="forward_accuracy")`, and the
    same call with `iv="dataset"`.
*   `xaikitTest.plot_results_grid(ivs=list(iv_config.keys()), dvs=list(dvs.keys()), phase="testing", title=...)`
    -- `iv_config.keys()` already includes `dataset`.

</details>

In [ ]:
# YOUR CODE HERE

In [ ]:
# Delete this cell after passing all tests. This is NOT for interviewees

for iv_name in ["xai_type", "dataset"]:
    analysis = xaikitTest.analyze_iv_dv(iv=iv_name, dv="forward_accuracy")
    print(f"--- {iv_name} x forward_accuracy ---")
    display(analysis.descriptives)

xaikitTest.plot_results_grid(
    ivs=list(iv_config.keys()),
    dvs=list(dvs.keys()),
    phase="testing",
    title="CoXAM virtual-participant results",
)

<details>
<summary>Click here to see the correct code</summary>

```python
for iv_name in ["xai_type", "dataset"]:
    analysis = xaikitTest.analyze_iv_dv(iv=iv_name, dv="forward_accuracy")
    print(f"--- {iv_name} x forward_accuracy ---")
    display(analysis.descriptives)

xaikitTest.plot_results_grid(
    ivs=list(iv_config.keys()),
    dvs=list(dvs.keys()),
    phase="testing",
    title="CoXAM virtual-participant results",
)
```

</details>

### Step 4.2 Two-IV interaction plot

**Goal:**

Show explanation presence against explanation family as a grouped bar chart.

**Instruction:**

`tested_w_xai` on the x axis, `xai_type` in the hue, `forward_accuracy` as the
value, testing phase, with readable labels for both factors.

Try it a second time with `hue_iv="dataset"` -- that view answers the question
this design was widened to ask: does the explanation effect replicate on the
other dataset, or was it a property of one?

<details>
<summary>Hint</summary>

Call `xaikitTest.plot_dv_by_two_ivs(...)` with `x_iv`, `hue_iv`, `dv`, `phase`, `x_levels`, `hue_levels`, `x_labels`, `hue_labels` and `title`.

</details>

In [ ]:
# YOUR CODE HERE

In [ ]:
# Delete this cell after passing all tests. This is NOT for interviewees

accuracy_plot = xaikitTest.plot_dv_by_two_ivs(
    x_iv="tested_w_xai",
    hue_iv="xai_type",
    dv="forward_accuracy",
    phase="testing",
    x_levels=[True, False],
    hue_levels=["decision_tree", "logistic_regression"],
    x_labels={True: "With explanation", False: "Without explanation"},
    hue_labels={
        "decision_tree": "Rules (Decision Tree)",
        "logistic_regression": "Weights (Logistic Regression)",
    },
    title="Forward accuracy by explanation presence and schema",
)
accuracy_plot.figure;

dataset_plot = xaikitTest.plot_dv_by_two_ivs(
    x_iv="xai_type",
    hue_iv="dataset",
    dv="forward_accuracy",
    phase="testing",
    x_levels=["decision_tree", "logistic_regression"],
    hue_levels=["wine_quality", "mushrooms"],
    x_labels={
        "decision_tree": "Rules (Decision Tree)",
        "logistic_regression": "Weights (Logistic Regression)",
    },
    hue_labels={"wine_quality": "Wine Quality", "mushrooms": "Mushrooms"},
    title="Does the explanation effect replicate across datasets?",
)
dataset_plot.figure;

<details>
<summary>Click here to see the correct code</summary>

```python
accuracy_plot = xaikitTest.plot_dv_by_two_ivs(
    x_iv="tested_w_xai",
    hue_iv="xai_type",
    dv="forward_accuracy",
    phase="testing",
    x_levels=[True, False],
    hue_levels=["decision_tree", "logistic_regression"],
    x_labels={True: "With explanation", False: "Without explanation"},
    hue_labels={
        "decision_tree": "Rules (Decision Tree)",
        "logistic_regression": "Weights (Logistic Regression)",
    },
    title="Forward accuracy by explanation presence and schema",
)
accuracy_plot.figure;

dataset_plot = xaikitTest.plot_dv_by_two_ivs(
    x_iv="xai_type",
    hue_iv="dataset",
    dv="forward_accuracy",
    phase="testing",
    x_levels=["decision_tree", "logistic_regression"],
    hue_levels=["wine_quality", "mushrooms"],
    x_labels={
        "decision_tree": "Rules (Decision Tree)",
        "logistic_regression": "Weights (Logistic Regression)",
    },
    hue_labels={"wine_quality": "Wine Quality", "mushrooms": "Mushrooms"},
    title="Does the explanation effect replicate across datasets?",
)
dataset_plot.figure;
```

</details>

### Step 4.3 Test pairwise condition differences

**Goal:**

Compare every crossed condition against every other with a Holm correction.

**Instruction:**

Run pairwise comparisons of `forward_accuracy` across `xai_type` x
`tested_w_xai` on the testing rows, paired within participant, Holm-corrected.

**Leave `dataset` out of `condition_cols`.** Adding it would cross every cell
with every dataset, multiplying the comparison count and making the Holm
correction far more conservative -- and a difference *between* datasets is not
an explanation effect anyway. See the closing note.

<details>
<summary>Hint</summary>

*   `from src.statistical_analyst import pairwise_condition_tests`
*   `condition_cols=["xai_type", "tested_w_xai"]`, `participant_col="participantId"`,
    `correction="holm"`.

</details>

In [ ]:
# YOUR CODE HERE

In [ ]:
# Delete this cell after passing all tests. This is NOT for interviewees

from src.statistical_analyst import pairwise_condition_tests

pairwise_results = pairwise_condition_tests(
    testing_results,
    value_col="forward_accuracy",
    condition_cols=["xai_type", "tested_w_xai"],
    participant_col="participantId",
    correction="holm",
)
display(pairwise_results.table.round(4))

<details>
<summary>Click here to see the correct code</summary>

```python
from src.statistical_analyst import pairwise_condition_tests

pairwise_results = pairwise_condition_tests(
    testing_results,
    value_col="forward_accuracy",
    condition_cols=["xai_type", "tested_w_xai"],
    participant_col="participantId",
    correction="holm",
)
display(pairwise_results.table.round(4))
```

</details>

## A note on reading these numbers

This run uses 4 participants per condition and 8 testing trials each, which is a
demonstration size, not a study size. Treat the p-values as a check that the
pipeline works end to end, not as a finding.

**A difference between datasets is not an explanation effect.** `wine_quality`
and `mushrooms` have different features, different class balance and different
task difficulty, and no participant sees both. If mushrooms scores lower, the
honest reading is "this is a harder task", not "explanations work worse here".
The question a multi-dataset design actually answers is the *replication* one:
does the with-vs-without-explanation gap point the same way on both? That is
what the second plot in Step 4.2 shows.

**Verify your condition column reaches the model.** CoXAM's answers differ by
`xai_type` because CoXAM reads that column. Rename the IV and the same analysis
still runs and still prints p-values -- they just stop meaning anything.